# Smoothing and interpolation from a Table

PyProBE now exposes continuous curve fitting directly on `Table` via
`Table.to_curve(...)`. You can pass SciPy interpolators such as
`PchipInterpolator`, `CubicSpline`, and `Akima1DInterpolator`, or smoothers
such as `make_smoothing_spline`. In each case, PyProBE returns a labelled
`Curve` object.


In [ ]:
%%capture
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.interpolate import (
    Akima1DInterpolator,
    CubicSpline,
    make_smoothing_spline,
)

import pyprobe

%matplotlib inline

We'll build a small synthetic `Table` with a noisy voltage signal.


In [ ]:
rng = np.random.default_rng(7)
time = np.linspace(0.0, 600.0, 40)
voltage = (
    3.1
    + 0.0015 * time
    + 0.03 * np.sin(time / 45.0)
    + rng.normal(0.0, 0.012, size=time.size)
)

table = pyprobe.Table(
    pl.DataFrame(
        {
            "Test Time / s": time,
            "Voltage / V": voltage,
        }
    )
)

table.data.head()

Any supported interpolator can now be applied directly from the `Table`. Each
fit returns a `Curve`, so the result is continuous rather than another discrete
`Table`.


In [ ]:
interpolation_curves = {
    "PCHIP": table.to_curve("Voltage / V", x="Test Time / s"),
    "Cubic spline": table.to_curve(
        "Voltage / V",
        x="Test Time / s",
        fit=CubicSpline,
    ),
    "Akima": table.to_curve(
        "Voltage / V",
        x="Test Time / s",
        fit=Akima1DInterpolator,
    ),
}

pl.DataFrame(
    {
        "Method": list(interpolation_curves),
        "Return type": [
            type(curve).__name__ for curve in interpolation_curves.values()
        ],
        "Recorded method": [
            curve.metadata.extras["curve_method"]
            for curve in interpolation_curves.values()
        ],
        "x column": [curve.columns.x.name for curve in interpolation_curves.values()],
        "y column": [curve.columns.y.name for curve in interpolation_curves.values()],
    }
)

In [ ]:
dense_time = np.linspace(time.min(), time.max(), 400)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(time, voltage, color="black", s=18, label="Original samples", zorder=3)

for label, curve in interpolation_curves.items():
    ax.plot(dense_time, curve(dense_time), label=label)

ax.set_xlabel("Test Time / s")
ax.set_ylabel("Voltage / V")
ax.set_title("Interpolating curves returned from Table.to_curve()")
ax.legend()
plt.show()

Smoothers use the same API. Here we fit a smoothing spline, which also returns
a `Curve`.


In [ ]:
smoothing_curve = table.to_curve(
    "Voltage / V",
    x="Test Time / s",
    fit=make_smoothing_spline,
    lam=0.5,
)

type(smoothing_curve), smoothing_curve.metadata.extras["curve_method"]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(time, voltage, color="black", s=18, label="Original samples", zorder=3)
ax.plot(
    dense_time, smoothing_curve(dense_time), color="tab:red", label="Smoothing spline"
)

ax.set_xlabel("Test Time / s")
ax.set_ylabel("Voltage / V")
ax.set_title("A smoothing spline returned as a Curve")
ax.legend()
plt.show()

Because the result is a `Curve`, you can evaluate it directly, differentiate it,
or sample it back onto a discrete grid.


In [ ]:
smoothed_table = smoothing_curve.to_table(dense_time)
gradient_curve = smoothing_curve.derivative()

print(type(smoothed_table).__name__)
print(type(gradient_curve).__name__)
smoothed_table.data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(dense_time, gradient_curve(dense_time), color="tab:green")

ax.set_xlabel(gradient_curve.columns.x.name)
ax.set_ylabel(gradient_curve.columns.y.name)
ax.set_title("Gradient of the smoothing spline")
plt.show()